# Artistic Telephone: export the reveal

One notebook, one output folder. Decode the prompts, then present **the HTML page** locally or put that same folder on the course website. No helper files, widgets, Replicate account, or running Python server are needed for the presentation.

Use the original selected PNGs exported by the current ST3GG student notebook, with the group folders and filenames intact. This exporter does not decode older Telephone formats; those images still appear, labelled **Prompt unavailable**.

[ST3GG](https://github.com/elder-plinius/ST3GG) supplies the decoder. The Python code and the HTML template are all inside this notebook.

## 1. Install once

Run this in your local Jupyter environment. Pillow is installed with ST3GG. Installation needs internet access; the export and the resulting page can run offline afterwards.

In [ ]:
%pip install -q "stegg==3.0.0"

## 2. Choose the folders

`FOLDER` is the downloaded class folder containing `Group_A`, `Group_B`, etc. Nested download folders are also accepted. Only the selected handoff filenames are included, not private `_try` images.

```text
Telephone_Reveal/
    Group_A/
        01.png
        01_02.png
        ...
    Group_B/
        ...
```

`OUTPUT` is a **new or empty folder**, separate from the input. Nothing already there will be replaced. For a second export, choose another name or move the previous output yourself. Windows paths can use `r"C:\Users\yourname\Downloads\Telephone_Reveal"`.

In [ ]:
FOLDER = r""  # Folder containing Group_A, Group_B, ...
OUTPUT = "telephone-reveal"

## 3. Export

Run the cell below once; there is nothing to edit inside it. It creates `index.html` and an `images/` folder. It copies the original PNG bytes without resizing or modifying your source files.

Check the printed warnings before publishing. Missing positions stay in their correct round; duplicate positions are flagged rather than silently choosing an image. An unreadable prompt never prevents a valid image from being shown. The notebook does not print the decoded prompts.

Then **double-click `index.html`**. Choose a group and chain, show one image at a time, reveal each prompt separately, and show the five-image summary. The arrow keys navigate; `P` reveals the prompt. Moving to another image hides its prompt again.

In [ ]:
if not FOLDER:
    print("Set FOLDER above, then rerun Export.")
else:
    import json
    import re
    from pathlib import Path
    
    from PIL import Image
    from steg_core import create_config, decode_text
    
    source = Path(FOLDER).expanduser().resolve()
    output = Path(OUTPUT).expanduser().resolve()
    if not source.is_dir():
        raise ValueError("FOLDER must contain your downloaded Group_A, Group_B, ... folders.")
    if source == output or source in output.parents or output in source.parents:
        raise ValueError("Keep the input and output folders separate, not inside each other.")
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise FileExistsError("OUTPUT is not empty. Choose a new folder; existing files are never replaced.")
    
    # Same fixed configuration as the student notebook. Only the prompt is embedded.
    steg = create_config(channels="RGB", bits=1, compress=True, strategy="interleaved")
    positions, groups, warnings = {}, set(), []
    for path in sorted(source.rglob("*")):
        if path.is_dir() and re.fullmatch(r"Group_[A-F]", path.name):
            groups.add(path.name[-1])
        if not path.is_file() or path.suffix.lower() != ".png":
            continue
        relative = path.relative_to(source)
        parents = [part for part in relative.parts[:-1] if re.fullmatch(r"Group_[A-F]", part)]
        if path.is_symlink() or not path.resolve().is_relative_to(source):
            warnings.append(f"{relative}: skipped linked file.")
            continue
        if len(parents) != 1 or not re.fullmatch(r"0[1-5](?:_0[1-5]){0,4}\.png", path.name):
            warnings.append(f"{relative}: skipped; use a group folder and the original handoff filename.")
            continue
        seats = [int(seat) for seat in path.stem.split("_")]
        if seats != [((seats[0] - 1 + step) % 5) + 1 for step in range(len(seats))]:
            warnings.append(f"{relative}: skipped; seats are not in clockwise order.")
            continue
        key = (parents[0][-1], seats[0], len(seats))
        positions.setdefault(key, []).append(path)
    if not positions:
        raise ValueError("No handoff PNGs found. Check FOLDER and the group folders/filenames.")
    
    chains, copied, decoded = [], 0, 0
    for group in sorted(groups):
        for start in range(1, 6):
            slides = []
            for round_number in range(1, 6):
                seats = [((start - 1 + step) % 5) + 1 for step in range(round_number)]
                name = "_".join(f"{seat:02d}" for seat in seats) + ".png"
                slide = {"round": round_number, "filename": name,
                         "src": None, "prompt": None, "note": "Image missing."}
                matches = positions.get((group, start, round_number), [])
                if len(matches) > 1:
                    slide["note"] = "Duplicate position. Resolve the copies and export again."
                    warnings.append(f"Group_{group}/{name}: duplicate position; no image chosen.")
                elif matches:
                    path = matches[0]
                    try:
                        if path.stat().st_size > 20 * 1024 * 1024:
                            raise ValueError("Image exceeds 20 MB")
                        data = path.read_bytes()
                        with Image.open(path) as image:
                            if image.format != "PNG" or image.width * image.height > 16_000_000:
                                raise ValueError("Use a PNG no larger than 16 megapixels")
                            image.load()
                            try:
                                prompt = decode_text(image, config=steg)
                                if not prompt.strip():
                                    raise ValueError("Empty prompt")
                                slide["prompt"] = prompt
                                decoded += 1
                            except Exception:
                                warnings.append(f"Group_{group}/{name}: prompt unavailable (not readable as ST3GG).")
                    except (OSError, ValueError, Image.DecompressionBombError) as error:
                        slide["note"] = "Image could not be opened."
                        warnings.append(f"Group_{group}/{name}: image unreadable ({type(error).__name__}).")
                    else:
                        relative_image = Path("images") / f"Group_{group}" / name
                        destination = output / relative_image
                        destination.parent.mkdir(parents=True, exist_ok=True)
                        destination.write_bytes(data)  # Original PNG bytes, never resized or re-encoded.
                        slide.update(src=relative_image.as_posix(), note="")
                        copied += 1
                slides.append(slide)
            chains.append({"label": f"Group {group} / Chain {start:02d}", "slides": slides})
    
    # Inline data and relative image paths work with file:// and under a Jekyll subpath.
    # Escape '<' so even a prompt containing '</script>' remains text, never executable HTML.
    payload = json.dumps({"chains": chains, "warnings": warnings}, ensure_ascii=True).replace("<", "\\u003c")
    page = r'''<!doctype html>
    <html lang="en">
    <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Artistic Telephone | Reveal</title>
    <style>
    :root{color-scheme:dark;--bg:#141618;--panel:#1f2326;--ink:#f1f0eb;--muted:#aeb5b8;--line:#42494c;--accent:#d5edbb}
    *{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.5 system-ui,sans-serif}
    main{max-width:1400px;margin:auto;padding:24px 32px 36px}header{display:flex;align-items:end;justify-content:space-between;gap:16px;margin-bottom:18px}
    h1{font-size:clamp(24px,3vw,36px);letter-spacing:-.04em;font-weight:600;line-height:1.15;margin:0}header p{margin:6px 0 0;color:var(--muted);font-size:14px}
    label{font-size:13px;color:var(--muted);display:grid;gap:5px}select,button{font:inherit;border:1px solid var(--line);border-radius:6px;padding:10px 14px;background:var(--panel);color:var(--ink)}
    button{cursor:pointer}button:hover:not(:disabled){border-color:var(--accent)}button:disabled{opacity:.38;cursor:default}:focus-visible{outline:3px solid var(--accent);outline-offset:3px}
    #next{background:var(--accent);color:#182119;border-color:var(--accent)}nav{display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:16px}#progress{margin-left:auto;color:var(--muted);font-size:14px}
    #stage{height:clamp(220px,52vh,620px);display:flex;align-items:center;justify-content:center;background:#090b0c;border:1px solid #2e3335;border-radius:8px}
    #hero{width:100%;height:100%;object-fit:contain}#empty{text-align:center;color:var(--muted);padding:24px}#filename{margin:10px 0;color:var(--muted);font-size:13px}
    #prompt,.card p{white-space:pre-wrap;overflow-wrap:anywhere}#prompt{background:var(--panel);border-left:3px solid var(--accent);padding:16px 20px;margin:14px 0 0}
    #summary{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:14px}.card{min-width:0;background:var(--panel);border:1px solid var(--line);border-radius:8px;overflow:hidden}
    .card img{display:block;width:100%;height:175px;object-fit:contain;background:#090b0c}.card h2{font-size:15px;margin:14px 14px 0}.card small{display:block;color:var(--muted);margin:4px 14px;overflow-wrap:anywhere}.card p{font-size:14px;margin:14px}
    #checks{margin-top:22px;color:var(--muted);font-size:13px}#checks ul{padding-left:22px}footer{margin-top:22px;font-size:12px;color:var(--muted)}[hidden]{display:none!important}
    @media(max-width:850px){main{padding:20px}header{align-items:start;flex-direction:column}#summary{grid-template-columns:repeat(2,minmax(0,1fr))}#progress{width:100%;margin-left:0}}
    @media(max-width:480px){#summary{grid-template-columns:1fr}button{padding:9px 10px;font-size:14px}}
    </style>
    </head>
    <body>
    <main>
    <header><div><h1>Artistic Telephone</h1><p>One image, another interpretation.</p></div>
    <label>Group and chain<select id="chain" aria-label="Group and chain"></select></label></header>
    <nav aria-label="Reveal controls">
    <button id="previous">Previous</button><button id="reveal" aria-expanded="false" aria-controls="prompt">Reveal prompt</button>
    <button id="next">Next image</button><button id="overview">Chain summary</button>
    <span id="progress" aria-live="polite"></span>
    </nav>
    <section id="single" aria-label="Current image"><div id="stage"><img id="hero" alt=""><p id="empty" hidden></p></div><p id="filename"></p><p id="prompt" hidden></p></section>
    <section id="summary" aria-label="All five images and prompts" hidden></section>
    <details id="checks" hidden><summary>Files to check before publishing</summary><ul id="warnings"></ul></details>
    <footer>Arrow keys: previous / next. P: reveal prompt. The summary shows all five prompts.<br>Publish only after the class reveal: prompts are already inside this page.</footer>
    <noscript>Enable JavaScript to navigate the reveal. No server or internet connection is needed.</noscript>
    </main>
    <script id="reveal-data" type="application/json">__REVEAL_DATA__</script>
    <script>
    const data = JSON.parse(document.getElementById('reveal-data').textContent);
    const $ = id => document.getElementById(id);
    let chainIndex = 0, position = 0;
    for (const [index, chain] of data.chains.entries()) $('chain').add(new Option(chain.label, index));
    for (const warning of data.warnings) { const li = document.createElement('li'); li.textContent = warning; $('warnings').append(li); }
    $('checks').hidden = !data.warnings.length;
    function render() {
      const chain = data.chains[chainIndex], summary = position === 5;
      $('chain').value = chainIndex;
      $('progress').textContent = chain.label + (summary ? ' / Complete chain' : ' / Round ' + (position + 1) + ' of 5');
      $('single').hidden = summary; $('summary').hidden = !summary;
      $('prompt').hidden = true; $('reveal').textContent = 'Reveal prompt'; $('reveal').setAttribute('aria-expanded', 'false');
      $('previous').disabled = chainIndex === 0 && position === 0;
      $('next').disabled = chainIndex === data.chains.length - 1 && summary;
      $('next').textContent = summary ? 'Next chain' : position === 4 ? 'Chain summary' : 'Next image';
      $('overview').disabled = summary;
      $('reveal').disabled = summary || !chain.slides[position].prompt;
      $('summary').replaceChildren();
      if (summary) {
        for (const slide of chain.slides) {
          const card = document.createElement('article'); card.className = 'card';
          if (slide.src) { const image = document.createElement('img'); image.src = slide.src; image.alt = slide.filename; card.append(image); }
          const title = document.createElement('h2'); title.textContent = 'Round ' + slide.round;
          const name = document.createElement('small'); name.textContent = slide.filename;
          const text = document.createElement('p'); text.textContent = slide.src ? (slide.prompt ?? 'Prompt unavailable.') : slide.note;
          card.append(title, name, text); $('summary').append(card);
        }
      } else {
        const slide = chain.slides[position];
        $('hero').hidden = !slide.src; $('empty').hidden = !!slide.src;
        if (slide.src) { $('hero').src = slide.src; $('hero').alt = chain.label + ', round ' + slide.round; }
        else $('hero').removeAttribute('src');
        $('empty').textContent = slide.note;
        $('filename').textContent = slide.filename + (slide.src && !slide.prompt ? ' / Prompt unavailable.' : '');
        $('prompt').textContent = slide.prompt ?? 'Prompt unavailable.';
      }
    }
    $('chain').onchange = () => { chainIndex = Number($('chain').value); position = 0; render(); };
    $('next').onclick = () => { if (position < 5) position++; else if (chainIndex < data.chains.length - 1) { chainIndex++; position = 0; } render(); };
    $('previous').onclick = () => { if (position > 0) position--; else if (chainIndex > 0) { chainIndex--; position = 5; } render(); };
    $('overview').onclick = () => { position = 5; render(); };
    $('reveal').onclick = () => { $('prompt').hidden = !$('prompt').hidden; $('reveal').textContent = $('prompt').hidden ? 'Reveal prompt' : 'Hide prompt'; $('reveal').setAttribute('aria-expanded', String(!$('prompt').hidden)); };
    document.addEventListener('keydown', event => {
      if (['SELECT', 'INPUT', 'TEXTAREA'].includes(event.target.tagName) || event.ctrlKey || event.metaKey || event.altKey) return;
      const button = {ArrowRight: 'next', ArrowLeft: 'previous', p: 'reveal', P: 'reveal'}[event.key];
      if (button && !$(button).disabled) { event.preventDefault(); $(button).click(); }
    });
    render();
    </script>
    </body>
    </html>
    '''
    output.mkdir(parents=True, exist_ok=True)
    (output / "index.html").write_text(page.replace("__REVEAL_DATA__", payload), encoding="utf-8")
    missing = sum(slide["src"] is None for chain in chains for slide in chain["slides"])
    print(f"Exported {copied} PNGs; {decoded} readable prompts; {missing} missing/duplicate/unreadable positions.")
    for warning in warnings:
        print("Check:", warning)
    print("Open in your browser:", output / "index.html")
    print("Move the WHOLE output folder to publish it. Source PNGs were not changed.")

## Publish the same folder with Jekyll

After the class reveal, copy the **whole** exported folder into the website, for example as `galleries/telephone/`. The generated HTML has no YAML front matter, uses relative image paths, and includes its own CSS, JavaScript, and decoded prompt data. There is no `fetch()`, CDN, external template, or Python build step.

Add this link in a Jekyll-processed course page:

```liquid
[Artistic Telephone - reveal and gallery]({{ '/galleries/telephone/index.html' | relative_url }})
```

Do not add front matter to the generated HTML or wrap it in a Jekyll layout. Jekyll treats it as a [static file](https://jekyllrb.com/docs/static-files/). The [`relative_url` filter](https://jekyllrb.com/docs/liquid/filters/#relative-url) keeps the course-page link compatible with a repository subpath.

**Publish only after the reveal and review the content first.** All decoded prompts are already in the HTML, even while hidden on screen. Do not upload private attempts, tokens, notebooks, or the source download folder. The original PNGs are copied at full size, so check the total gallery size before committing it.